# Agent vs PID (stage 2, moving pad 0.2-0.5 m/s)

Fair comparison between the PPO agents and the PID baseline. Both get their settings chosen on validation seeds
500-599; the test seeds 1000-1099 are only used once afterwards. Reads only CSVs:
`results/<run>/val/*.csv` (checkpoints on validation seeds, `evaluate.py`), the test `eval_*.csv` files and
`results/pid/tradeoff_val*.csv` (`pid_sweep.py`).

Runs 3a/3b see the 12-value observation; run 4 also sees the drone's own velocity (15 values), so it can work out the
pad's velocity. Each agent is compared with the PID that has the same information.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("../results")
RUNS = {"run3a_5M": "3a (12 values)", "run3b_gamma0995": "3b (12 values, gamma 0.995)",
        "run4_ownvel": "4 (15 values, own velocity)"}
# Highest validation success per run, ties -> lower touchdown speed (see the selection table below)
CHOSEN = {"run3a_5M": 4_100_000, "run4_ownvel": 4_700_000}
PARAMS = ["kp", "align_radius", "descent_gain", "max_descent", "min_descent"]

In [ ]:
# Chart style: light surface, recessive grid and axes, thin lines
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#898781", "ytick.color": "#898781", "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})

# Fixed colour per policy, the same in every chart
COLORS = {"run3a_5M": "#2a78d6", "pid": "#eb6834", "run4_ownvel": "#1baf7a", "run3b_gamma0995": "#eda100"}
MUTED = "#b5b4ad"

In [ ]:
def summarise(df: pd.DataFrame) -> dict:
    # Time to land = median duration of successful episodes, the same for agent and PID
    touched = df[df["outcome"].isin(["landed", "bad_landing"])]
    return {
        "success": df["success"].mean(),
        "time_s": df.loc[df["success"], "duration_s"].median(),
        "touchdown_speed": touched["touchdown_speed"].median(),
        "offset_cm": 100 * touched["touchdown_offset"].median(),
    }

## Checkpoint selection on validation seeds 500-599

Checkpoints every 100k steps from 2.5M (all runs are out of the hover trap by then), 26 per run, so every run gets
the same number of candidates. Rule: highest validation success, ties broken by lower median touchdown speed.
Taking the best of 26 noisy scores flatters the winner, so its validation score is not the result; the single test
run below is.

In [ ]:
rows = []
for run in RUNS:
    for path in (RESULTS / run / "val").glob("model_*_steps.csv"):
        step = int(re.search(r"model_(\d+)_steps", path.name).group(1))
        rows.append({"run": run, "step": step, **summarise(pd.read_csv(path))})
val = pd.DataFrame(rows).sort_values(["run", "step"])
val.sort_values(["success", "touchdown_speed"], ascending=[False, True]).groupby("run").head(3).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for run, label in RUNS.items():
    d = val[val["run"] == run]
    ax.plot(d["step"] / 1e6, 100 * d["success"], color=COLORS[run], label=label)
chosen = {run: val[(val["run"] == run) & (val["step"] == step)].iloc[0] for run, step in CHOSEN.items()}
for run, row in chosen.items():
    ax.plot(row["step"] / 1e6, 100 * row["success"], "o", markersize=9, color=COLORS[run],
            markeredgecolor="#fcfcfb", markeredgewidth=2)
    ax.annotate(f"chosen: {row['step'] / 1e6:g}M, {row['success']:.0%}", (row["step"] / 1e6, 100 * row["success"]),
                xytext=(0, 10), textcoords="offset points", ha="center", color="#52514e",
                bbox={"boxstyle": "round,pad=0.2", "facecolor": "#fcfcfb", "edgecolor": "none"})
ax.set_ylim(top=106)
ax.set_xlabel("training steps (millions)")
ax.set_ylabel("validation success (%)")
ax.set_title("Validation success per checkpoint")
ax.legend(loc="lower right", ncols=3)
plt.show()

Success jumps by several points between neighbouring checkpoints (100k steps = 6 PPO updates). Even the final model
and the 5M checkpoint, one PPO update apart, differ by 5 points (91 vs 96%). So a single checkpoint's score has a
few points of noise, and checkpoint choice matters.

## Test seeds 1000-1099 (run once)

In [ ]:
tests = {
    "3a final (5.01M)": RESULTS / "run3a_5M/eval_seed1000_pad0.2-0.5.csv",
    "3a 4.1M (chosen on validation)": RESULTS / "run3a_5M/eval_seed1000_pad0.2-0.5_4100k.csv",
    "3b final (5.01M)": RESULTS / "run3b_gamma0995/eval_seed1000_pad0.2-0.5.csv",
    "4 final (5.01M)": RESULTS / "run4_ownvel/eval_seed1000_pad0.2-0.5.csv",
    "4 4.7M (chosen on validation)": RESULTS / "run4_ownvel/eval_seed1000_pad0.2-0.5_4700k.csv",
    "PID (tuned, 12 values)": RESULTS / "pid/eval_seed1000_pad0.2-0.5.csv",
    "PID feedforward (tuned, 15 values)": RESULTS / "pid/eval_seed1000_pad0.2-0.5_feedforward.csv",
}
test = {}
for name, path in tests.items():
    df = pd.read_csv(path)
    by_speed = df.groupby(pd.cut(df["pad_speed"], [0.2, 0.3, 0.4, 0.5]), observed=True)["success"].mean()
    test[name] = {**summarise(df), **{f"success {b.left:g}-{b.right:g}": v for b, v in by_speed.items()}}
pd.DataFrame(test).T.round(3)

## Hover trap: when does each run start landing?

From the Monitor CSVs (one row per training episode, stochastic policy). Steps are approximate: each env's episode
lengths are summed and multiplied by the 8 parallel envs.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for run, label in RUNS.items():
    parts = []
    for path in (RESULTS / run / "monitor").glob("*.monitor.csv"):
        episodes = pd.read_csv(path, skiprows=1)
        episodes["steps"] = episodes["l"].cumsum() * 8
        parts.append(episodes)
    episodes = pd.concat(parts)
    landed = episodes.groupby(episodes["steps"] // 100_000 * 0.1)["outcome"].apply(lambda o: (o == "landed").mean())
    ax.plot(landed.index, 100 * landed, color=COLORS[run], label=label)
ax.set_xlabel("training steps (millions)")
ax.set_ylabel("landed in training (%)")
ax.set_title("Escape from the hover trap")
ax.legend(loc="lower right")
plt.show()

## PID speed-success trade-off (validation seeds 500-599)

The tuned PID is patient: it only descends when within 15 cm of the pad centre and slows down near the ground. The
sweep (`pid_sweep.py`, 72 configs) makes it more aggressive: higher KP, descending earlier (larger align radius) and
faster. The front connects the configs that no other config beats on both success and time.

Two PIDs: the plain one sees the same 12 values as runs 1-3 and has to learn the pad velocity with its I term.
The feedforward one (`--feedforward`, KI 0) sees the 15-value observation of run 4 and commands the pad velocity directly.
Caveat: time is measured on successful episodes only, so for low-success configs it is the time of the easy episodes.

In [ ]:
def pareto_front(configs: pd.DataFrame) -> pd.DataFrame:
    # Walking from fast to slow, keep a config only if it beats the success of every faster one
    front, best = [], -1.0
    for _, row in configs.sort_values(["time_s", "success"], ascending=[True, False]).iterrows():
        if row["success"] > best:
            front.append(row)
            best = row["success"]
    return pd.DataFrame(front)


sweeps = {}
for name, file in {"plain": "tradeoff_val.csv", "feedforward": "tradeoff_val_feedforward.csv"}.items():
    sweep = pd.read_csv(RESULTS / "pid" / file)
    sweeps[name] = pd.DataFrame([{**dict(zip(PARAMS, key)), **summarise(df)} for key, df in sweep.groupby(PARAMS)])
fronts = {name: pareto_front(configs) for name, configs in sweeps.items()}
pd.concat(fronts, names=["pid"]).round(3)

In [ ]:
TUNED = {"kp": 1.2, "align_radius": 0.15, "descent_gain": 0.6, "max_descent": 0.5, "min_descent": 0.15}
# Plain PID: filled circles and a solid front; feedforward PID: hollow triangles and a dashed front
STYLE = {"plain": {"marker": "o", "linestyle": "-", "label": "12 values"},
         "feedforward": {"marker": "^", "linestyle": "--", "label": "feedforward, 15 values"}}

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, configs in sweeps.items():
    ax.plot(configs["time_s"], 100 * configs["success"], STYLE[name]["marker"], markersize=6, color=MUTED,
            markerfacecolor=MUTED if name == "plain" else "none", label=f"PID configs, {STYLE[name]['label']}")
for name, front in fronts.items():
    ax.plot(front["time_s"], 100 * front["success"], marker=STYLE[name]["marker"], linestyle=STYLE[name]["linestyle"],
            markersize=9, color=COLORS["pid"], markeredgecolor="#fcfcfb", markeredgewidth=2,
            label=f"PID front, {STYLE[name]['label']}")
    tuned = sweeps[name][(sweeps[name][list(TUNED)] == pd.Series(TUNED)).all(axis=1)].iloc[0]
    ax.plot(tuned["time_s"], 100 * tuned["success"], "s", markersize=9, color=COLORS["pid"],
            markeredgecolor="#fcfcfb", markeredgewidth=2)
    ax.annotate(f"tuned PID ({name})", (tuned["time_s"], 100 * tuned["success"]), xytext=(0, -18),
                textcoords="offset points", ha="center", color="#52514e")
fastest_ff = fronts["feedforward"].iloc[-1]
ax.annotate(f"fastest feedforward PID: {fastest_ff['success']:.0%} at {fastest_ff['time_s']:.1f} s",
            (fastest_ff["time_s"], 100 * fastest_ff["success"]), xytext=(18, 12), textcoords="offset points",
            color="#52514e")
for run, row in chosen.items():
    ax.plot(row["time_s"], 100 * row["success"], "D", markersize=9, color=COLORS[run], markeredgecolor="#fcfcfb",
            markeredgewidth=2, label=f"agent {RUNS[run]}, {row['step'] / 1e6:g}M")
ax.set_ylim(top=106)
ax.set_xlabel("median time to land, successful episodes (s)")
ax.set_ylabel("validation success (%)")
ax.set_title("PID speed-success trade-off vs the agents")
ax.legend(loc="lower right")
plt.show()